In [2]:
import torch
from build_vocab import WordVocab
from pretrain_trfm import TrfmSeq2seq
from utils import split
import json
from transformers import T5EncoderModel, T5Tokenizer
import re
import gc
from sklearn import metrics
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from scipy import stats
from sklearn.model_selection import train_test_split
import random
import pickle
import math


C:\Users\memre\anaconda3\envs\Uni_test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def smiles_to_vec(Smiles):
    pad_index = 0
    unk_index = 1
    eos_index = 2
    sos_index = 3
    mask_index = 4
    vocab = WordVocab.load_vocab('vocab.pkl')
    def get_inputs(sm):
        seq_len = 220
        sm = sm.split()
        if len(sm)>218:
            # print('SMILES is too long ({:d})'.format(len(sm)))
            sm = sm[:109]+sm[-109:]
        ids = [vocab.stoi.get(token, unk_index) for token in sm]
        ids = [sos_index] + ids + [eos_index]
        seg = [1]*len(ids)
        padding = [pad_index]*(seq_len - len(ids))
        ids.extend(padding), seg.extend(padding)
        return ids, seg
    def get_array(smiles):
        x_id, x_seg = [], []
        for sm in smiles:
            a,b = get_inputs(sm)
            x_id.append(a)
            x_seg.append(b)
        return torch.tensor(x_id), torch.tensor(x_seg)
    trfm = TrfmSeq2seq(len(vocab), 256, len(vocab), 4)
    trfm.load_state_dict(torch.load('trfm_12_23000.pkl', map_location=torch.device('cpu')))
    trfm.eval()
    x_split = [split(sm) for sm in Smiles]
    xid, xseg = get_array(x_split)
    X = trfm.encode(torch.t(xid))
    return X


def Seq_to_vec(Sequence):
    for i in range(len(Sequence)):
        if len(Sequence[i]) > 1000:
            Sequence[i] = Sequence[i][:500] + Sequence[i][-500:]
    sequences_Example = []
    for i in range(len(Sequence)):
        zj = ''
        for j in range(len(Sequence[i]) - 1):
            zj += Sequence[i][j] + ' '
        zj += Sequence[i][-1]
        sequences_Example.append(zj)
    tokenizer = T5Tokenizer.from_pretrained("prot_t5_xl_uniref50", do_lower_case=False)
    model = T5EncoderModel.from_pretrained("prot_t5_xl_uniref50")
    gc.collect()
    print(torch.cuda.is_available())
    # 'cuda:0' if torch.cuda.is_available() else
    device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model = model.eval()
    features = []
    for i in range(len(sequences_Example)):
        print('For sequence ', str(i+1))
        sequences_Example_i = sequences_Example[i]
        sequences_Example_i = [re.sub(r"[UZOB]", "X", sequences_Example_i)]
        ids = tokenizer.batch_encode_plus(sequences_Example_i, add_special_tokens=True, padding=True)
        input_ids = torch.tensor(ids['input_ids']).to(device)
        attention_mask = torch.tensor(ids['attention_mask']).to(device)
        with torch.no_grad():
            embedding = model(input_ids=input_ids, attention_mask=attention_mask)
        embedding = embedding.last_hidden_state.cpu().numpy()
        for seq_num in range(len(embedding)):
            seq_len = (attention_mask[seq_num] == 1).sum()
            seq_emd = embedding[seq_num][:seq_len - 1]
            features.append(seq_emd)
    features_normalize = np.zeros([len(features), len(features[0][0])], dtype=float)
    for i in range(len(features)):
        for k in range(len(features[0][0])):
            for j in range(len(features[i])):
                features_normalize[i][k] += features[i][j][k]
            features_normalize[i][k] /= len(features[i])
    return features_normalize



    

In [3]:
def Kcat_predict(Ifeature, Label):
    kf = KFold(n_splits=5, shuffle=True)
    All_pre_label = []
    All_real_label = []
    for train_index, test_index in kf.split(Ifeature, Label):
        Train_data, Train_label = Ifeature[train_index], Label[train_index]
        Test_data, Test_label = Ifeature[test_index], Label[test_index]
        model = ExtraTreesRegressor()
        model.fit(Train_data, Train_label)
        Pre_label = model.predict(Test_data)
        All_pre_label.extend(Pre_label)
        All_real_label.extend(Test_label)
    res = pd.DataFrame({'Value': All_real_label, 'Predict_Label': All_pre_label})
    res.to_excel('Kcat_Km_5_cv.xlsx')

In [4]:
res = np.array(pd.read_excel('datasets/kcat_km_samples.xlsx', sheet_name='main')).T
Smiles = res[1]
sequences = res[2]
Value = res[0]
for i in range(len(Value)):
    Value[i] = math.log(Value[i], 10)
print(max(Value), min(Value))

8.968482948553934 -4.823908740944319


In [5]:
with open("Kcat_Km_features_910.pkl", "rb") as f:
    feature = pickle.load(f)

In [6]:
feature = np.array(feature)
Label = np.array(Value)
Kcat_predict(feature, Label)

In [8]:
df_y_values = pd.read_excel('Kcat_Km_5_cv.xlsx')

In [10]:
print('R2 is ' + str(r2_score(df_y_values['Value'], df_y_values['Predict_Label'])))
print('RMSE is ' + str(mean_squared_error(df_y_values['Value'], df_y_values['Predict_Label'], squared=False)))
print('PCC is ' + str(stats.pearsonr(df_y_values['Value'], df_y_values['Predict_Label'])[0]))

R2 is 0.6334908946914455
RMSE is 1.1059785923399676
PCC is 0.7963366641462515


In [12]:
print(np.shape(feature))
print(np.shape(Label))

(910, 2048)
(910,)


In [13]:
with open('UniKP for kcat_Km.pkl', "rb") as f:
    model = pickle.load(f)

In [14]:
Predicted_label = model.predict(feature)
print('R2 is ' + str(r2_score(Label, Predicted_label)))

R2 is 0.999995503898191


In [16]:
#beta-Glucosidase Dataset

Smiles = ['C1=CC(=CC=C1[N+](=O)[O-])O[C@H]2[C@@H]([C@H]([C@@H]([C@H](O2)CO)O)O)O'] #pNP-Glc
filepath = 'sequence_vec.xlsx'
df_seq_vec= pd.read_excel(filepath, index_col=None )
smiles_updated = np.shape(df_seq_vec)[0] * Smiles
smiles_vec = smiles_to_vec(smiles_updated)

fused_vector_BGL = np.concatenate((smiles_vec, df_seq_vec.values), axis=1)

C:\Users\memre\anaconda3\envs\Uni_test\lib\site-packages\torch\nn\modules\transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
C:\Users\memre\AppData\Local\Temp\ipykernel_2280\870178684.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer

There are 258 molecules. It will take a little time.


In [3]:
def removeoutlier_col(df,cols):
    Q1 = df[cols].quantile(0.25)
    Q3 = df[cols].quantile(0.75)
    IQR = Q3 - Q1
    df_out = df[~((df[[cols]] < (Q1 - 1.5 * IQR)) |(df[[cols]] > (Q3 + 1.5 * IQR))).any(axis=1)]
    return df_out

df = pd.read_excel('betaGlucosidasewithMutantsOptimumTemperature.xlsx')
output = 'pNP-Glc kcat/Km (1/smM)'
df['Log'+output] = np.log10(df[output])
df_clean = removeoutlier_col(df,'Log' + output).reset_index()
BGL_Label = df_clean[df_clean['Percentage Activity Depending on Optimum Temp']==1]['Log'+output]
sequence_list = df_clean[df_clean['Percentage Activity Depending on Optimum Temp']==1]['Sequence'].values

In [24]:
print(np.shape(fused_vector_BGL))
print(np.shape(BGL_Label))

(258, 2048)
(258,)


In [25]:
Predicted_BGL = model.predict(fused_vector_BGL)

In [26]:
res = pd.DataFrame({'sequences': sequence_list, 'Value':BGL_Label ,
                    'Predicted_label': Predicted_BGL})
res.to_excel('20250827 Kinetic_parameters_predicted_label.xlsx')

In [31]:
print('R2 is ' + str(r2_score(BGL_Label, Predicted_BGL)))
print('RMSE is ' + str(mean_squared_error(BGL_Label, Predicted_BGL, squared=False)))
print('MAE is ' + str(mean_absolute_error(BGL_Label, Predicted_BGL)))
print('PCC is ' + str(stats.pearsonr(BGL_Label, Predicted_BGL)[0]))
print('p value is ' + str(stats.pearsonr(BGL_Label, Predicted_BGL)[1]))

R2 is 0.051183087177614994
RMSE is 1.4629519101085082
MAE is 1.2212486950875034
PCC is 0.30081711050464777
p value is 8.528290840093074e-07


In [9]:
df_clean[df_clean['Percentage Activity Depending on Optimum Temp']==1].to_excel('BGL_outliers_removed.xlsx')